In [18]:
import os
import io

from typing import Optional, List

from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

import fitz
from PIL import Image

In [19]:

class Dimensions(BaseModel):
    width_mm: Optional[float] = None
    depth_mm: Optional[float] = None
    height_mm: Optional[float] = None


class Installation(BaseModel):
    type: Optional[str] = None
    waste_outlet: Optional[str] = None
    rough_in_mm: Optional[float] = None


class Electrical(BaseModel):
    required: Optional[bool] = None
    voltage: Optional[str] = None
    power_w: Optional[float] = None


class Product(BaseModel):
    product_id: Optional[str] = None
    product_name: Optional[str] = None
    category: Optional[str] = None
    subcategory: Optional[str] = None
    collection: Optional[str] = None

    dimensions: Dimensions = Field(default_factory=Dimensions)
    installation: Installation = Field(default_factory=Installation)
    electrical: Electrical = Field(default_factory=Electrical)

    features: List[str] = Field(default_factory=list)
    color: List[str] = Field(default_factory=list)

    material: Optional[str] = None

In [20]:
load_dotenv()


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a product data extraction system for a
KOHLER bathroom product catalog.

Extract ONLY information explicitly present in the
provided document.

Rules:
- Never invent or guess information.
- Missing information must be null.
- Preserve product/model numbers exactly.
- Convert dimensions to millimetres when unambiguous.
- Convert power to watts when unambiguous.
- Extract explicitly stated features.
- Keep installation information separate.
- Extract electrical requirements only when stated.
- Do not infer price.
- Do not infer aesthetic/theme scores.
- Do not make recommendations.
- Do not decide whether a product fits a bathroom.
- Use only information contained in the document.
"""
    ),
    (
        "human",
        """
Extract the product information from this document:

{document}
"""
    )
])


structured_llm = llm.with_structured_output(Product)

chain = prompt | structured_llm

In [21]:
def Extractor(pdf):
    loader = PyPDFLoader(pdf)
    pages = loader.load()

    text = "\n".join(
        doc.page_content
        for doc in pages
    )

    product = chain.invoke({
        "document": text
    })

    return product

    
    

    

In [30]:
pdf_path = "C:\\Users\\Abhist\\Desktop\\KOHLER\\data\\rawData\\K-28529IN_spec_IN_Kohler_en.pdf"
product = Extractor(pdf_path)
print(product.model_dump_json(indent=2))


{
  "product_id": "K-28529IN",
  "product_name": "Leap™ One-piece round-front smart toilet, dual-flush",
  "category": "Toilet",
  "subcategory": "One-piece round-front smart toilet",
  "collection": "Leap™",
  "dimensions": {
    "width_mm": 369.0,
    "depth_mm": 700.0,
    "height_mm": 400.0
  },
  "installation": {
    "type": "Floor-mount",
    "waste_outlet": "Floor",
    "rough_in_mm": 305.0
  },
  "electrical": {
    "required": true,
    "voltage": "220-240 V",
    "power_w": 1200.0
  },
  "features": [
    "One-piece design integrates tank and bowl",
    "Round-front bowl for smaller bathrooms",
    "Dual-flush 0.8 or 1.2 gpf (3.0 or 4.5 lpf)",
    "Touchless technology for hands-free flush actuation",
    "Automatic flushing after each use",
    "Remote control and backup button operation",
    "Manual flush button on control panel",
    "Fully skirted trapway for easy cleaning",
    "1.9 inch (49 mm) fully glazed trapway",
    "Includes bidet seat",
    "Quiet-Close™ seat f

In [23]:
def pdf_page_to_image(pdf_path, page_number):

    document = fitz.open(pdf_path)

    page = document[page_number]

    # Increase resolution
    matrix = fitz.Matrix(2, 2)

    pix = page.get_pixmap(matrix=matrix)

    image_bytes = pix.tobytes("png")

    image = Image.open(
        io.BytesIO(image_bytes)
    )

    document.close()

    return image

In [25]:
document = fitz.open(pdf_path)

print("Number of pages:", len(document))

document.close()

Number of pages: 2


In [26]:
class ExtractedDimensions(BaseModel):

    width_mm: Optional[float] = None
    depth_mm: Optional[float] = None
    height_mm: Optional[float] = None

In [27]:
dimension_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are extracting physical dimensions from a
manufacturer's technical drawing.

Identify ONLY dimensions explicitly shown in the
technical drawing.

Extract:

- width
- depth
- height

Rules:

1. Do not guess.
2. Do not use general product knowledge.
3. Do not assume typical toilet dimensions.
4. Use only dimensions explicitly visible in the drawing.
5. Return dimensions in millimetres.
6. If a dimension cannot be confidently identified,
   return null.
7. Ignore dimensions that are not the overall product
   width, depth, or height.
"""
    ),
    (
        "human",
        "Analyze the technical drawing and extract the dimensions."
    )
])

In [28]:
def ValidateProduct(product):

    problems = []

    if not product.product_id:
        problems.append("Missing product ID")

    if not product.product_name:
        problems.append("Missing product name")

    if not product.category:
        problems.append("Missing category")

    if product.dimensions.width_mm is None:
        problems.append("Missing width")

    if product.dimensions.depth_mm is None:
        problems.append("Missing depth")

    if product.dimensions.height_mm is None:
        problems.append("Missing height")

    if product.installation.rough_in_mm is None:
        problems.append("Missing rough-in")

    return problems

In [29]:
problems = ValidateProduct(product)

if problems:
    print("Problems found:")

    for problem in problems:
        print("-", problem)

else:
    print("Product passed validation")

Problems found:
- Missing width
- Missing depth
